# 03 · Ranking de estados y participación en el consumo total

Práctica basada en `notebooks/02_energia.ipynb`. Tiempo sugerido: 20–30 minutos.

**Tu tarea:** completá únicamente la función de la sección 3.
La carga de datos, el entorno y las pruebas ya están preparados.

1. Elegí el kernel del entorno `.venv` del proyecto y ejecutá las celdas de preparación.
2. Leé el contrato y reemplazá el `raise NotImplementedError` por tu implementación.
3. Ejecutá de nuevo la celda de la función y después la de pruebas.
4. Si todas pasan, esa misma celda probará tu función sobre los datos reales y mostrará el resultado.

Podés usar **Ejecutar todo** desde un kernel nuevo. Mientras la función esté pendiente,
la última celda mostrará un aviso; no tenés que instalar nada adicional en el entorno actual.

## 1. Preparación lista

La base equivale a `base` del notebook original: fecha como índice, nombres preparados y faltantes, negativos y ceros conservados.

Los archivos de `../datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_energia()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 285,499 | Columnas: 13


## 2. Consigna y resultado esperado

Sumá el `Consumo` de todo el período por `UF` y devolvé los `n` estados con mayor total.

- Usá `sum(min_count=1)` y omití del ranking las UF sin ninguna medición de consumo.
- Devolvé un DataFrame con índice `UF` y columnas `consumo_total`, `participacion_pct`.
- La participación es el consumo de la UF dividido por el total de **todas las UF con datos**,
  multiplicado por 100. No uses solo la suma del top `n` como denominador.
- Ordená por consumo descendente; si hay empate, por nombre de UF ascendente.
- Conservá los valores negativos. Si el total general es cero, las participaciones son `NaN`.
- `n` es un entero positivo. Si hay menos UF, devolvé las disponibles.
  No redondees ni modifiques la entrada.

## 3. Tu función

Editá solo esta celda. Mantené el nombre de la función y sus parámetros para que funcionen las pruebas.

In [3]:
def ranking_estados(datos, n=5):
    """Implementá el contrato de la sección 2 sin modificar datos."""
    # TODO: escribí tu solución acá.
    raise NotImplementedError("Completá esta función y volvé a ejecutar las pruebas.")

## 4. Pruebas y ejecución real

Un `AssertionError` indica qué resultado difiere del esperado. Primero revisá el caso pequeño que falló. Después de editar la función, ejecutá su celda antes de volver a correr estas pruebas.

Las pruebas verifican ejemplos pequeños y propiedades del resultado real; no sustituyen una revisión de tu razonamiento.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    caso = pd.DataFrame({
        "UF": ["SP", "RJ", "SP", "MG", "RS", "SC", "RS"],
        "Consumo": [100, 150, 50, 50, np.nan, -10, np.nan],
    })
    original = caso.copy(deep=True)
    esperado = pd.DataFrame({
        "consumo_total": [150, 150, 50, -10],
        "participacion_pct": [150 / 340 * 100, 150 / 340 * 100, 50 / 340 * 100, -10 / 340 * 100],
    }, index=pd.Index(["RJ", "SP", "MG", "SC"], name="UF"))
    comparar(funcion(caso, n=2), esperado.iloc[:2])
    comparar(funcion(caso, n=99), esperado)
    comparar(funcion(caso), esperado)

    total_cero = pd.DataFrame({"UF": ["SP", "RJ"], "Consumo": [10, -10]})
    esperado_cero = pd.DataFrame({
        "consumo_total": [10, -10], "participacion_pct": [np.nan, np.nan],
    }, index=pd.Index(["SP", "RJ"], name="UF"))
    comparar(funcion(total_cero), esperado_cero)

    sin_datos = pd.DataFrame({"UF": ["SP"], "Consumo": [np.nan]})
    comparar(funcion(sin_datos), esperado.iloc[:0])
    comparar(caso, original)
    print("OK: sumas, empates, denominador global, negativos, n variable y total cero.")


resultado = None
try:
    probar(ranking_estados)
except NotImplementedError:
    print("PENDIENTE: completá la función de la sección 3 y volvé a ejecutar esta celda.")
else:
    # Comprobamos también que el dataset real permanezca intacto.
    antes = datos.copy(deep=True)
    resultado = ranking_estados(datos)
    comparar(datos, antes)
    assert len(resultado) == min(5, datos.loc[datos["Consumo"].notna(), "UF"].nunique())
    assert resultado["consumo_total"].is_monotonic_decreasing
    np.testing.assert_allclose(
        resultado["participacion_pct"], resultado["consumo_total"] / datos["Consumo"].sum() * 100,
    )
    print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
    display(resultado.head(20))

PENDIENTE: completá la función de la sección 3 y volvé a ejecutar esta celda.


## Pista opcional

<details>
<summary>Abrir solo si te trabás</summary>

Calculá el total general antes de aplicar `head(n)`. Para empates, pensá en dos criterios de orden.

</details>

**Para explicar en voz alta:** ¿qué filas entran al cálculo?, ¿qué representa cada fila de la salida?, ¿qué caso límite cubre cada prueba?